# 📋 Struttura Completa del Notebook

Sezioni Principali:

1. Caricamento Dati - Carica il CSV preprocessato
2. Setup Preprocessing - Definisce transformer per encoding
3. Split Train-Test - Divisione stratificata 80/20

*5 Approcci Testati:*

- BASELINE - Solo class weighting (modello attuale)
- Tomek Links - Pulisce boundary decisionale
- NearMiss - Seleziona esempi difficili
- Random Undersampling + Ensemble - 5 modelli con voting
- SMOTE + Undersampling - Hybrid approach

*Confronto Finale Include:*

- Tabella comparativa formattata
-  Migliore per ogni metrica singola
- Score aggregato pesato (Recall 30%, Precision 25%, F1 25%, ROC-AUC 20%)
- Confronto dettagliato con baseline
- Raccomandazione automatica con emoticon
- Classifica completa con medaglie 🥇🥈🥉
- Esportazione risultati in CSV

In [1]:
# pip install imbalanced-learn

In [18]:
"""
NOTEBOOK: CONFRONTO APPROCCI PER GESTIONE CLASSI SBILANCIATE
==============================================================

OBIETTIVO: Testare 4 diversi approcci di resampling e confrontarli con la baseline
           (solo class weighting) per determinare se possiamo migliorare le performance.

PREREQUISITI:
- CSV preprocessato con outlier rimossi (8183 righe, 16 colonne)
- Librerie: pandas, numpy, sklearn, imbalanced-learn

STRUTTURA:
1. Caricamento dati
2. Split train-test
3. Baseline (solo class weighting)
4. Approccio 1: Tomek Links
5. Approccio 2: NearMiss
6. Approccio 3: Random Undersampling + Ensemble
7. Approccio 4: SMOTE + Undersampling
8. Confronto finale e raccomandazione
"""

import pandas as pd
import numpy as np
import time
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, auc, balanced_accuracy_score, 
                             f1_score, recall_score, precision_score)
from sklearn.utils.class_weight import compute_class_weight

# Librerie per resampling (installa con: pip install imbalanced-learn)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks, NearMiss, RandomUnderSampler

print("="*80)
print("NOTEBOOK: CONFRONTO APPROCCI BILANCIAMENTO DATASET")
print("="*80)
print("\nLibrerie caricate con successo!\n")

# ==========================================
# SEZIONE 1: CARICAMENTO DATI
# ==========================================
print("="*80)
print("SEZIONE 1: CARICAMENTO DATI")
print("="*80)

# IMPORTANTE: Modifica il percorso se necessario
CSV_PATH = './BunkerChurners_PearsonCleaned_OutliersRemoved.csv'

print(f"\nCaricamento CSV da: {CSV_PATH}")
df = pd.read_csv(CSV_PATH)

print(f"✓ Dataset caricato: {df.shape}")
print(f"\nPrime righe:")
print(df.head())

print(f"\nDistribuzione target (Attrition_Flag):")
print(df['Attrition_Flag'].value_counts())
print(f"\nRapporto di sbilanciamento: {df['Attrition_Flag'].value_counts().iloc[0] / df['Attrition_Flag'].value_counts().iloc[1]:.2f}:1")

NOTEBOOK: CONFRONTO APPROCCI BILANCIAMENTO DATASET

Librerie caricate con successo!

SEZIONE 1: CARICAMENTO DATI

Caricamento CSV da: ./BunkerChurners_PearsonCleaned_OutliersRemoved.csv
✓ Dataset caricato: (8183, 16)

Prime righe:
      Attrition_Flag  Dependent_count Education_Level Marital_Status  \
0  Existing Customer                5      Uneducated        Unknown   
1  Existing Customer                2        Graduate        Married   
2  Existing Customer                2        Graduate        Married   
3  Existing Customer                1       Doctorate       Divorced   
4  Attrited Customer                0        Graduate        Married   

  Income_Category Card_Category  Months_on_book  Total_Relationship_Count  \
0         $120K +          Blue              31                         5   
1  Less than $40K          Blue              48                         5   
2         Unknown          Blue              37                         6   
3     $60K - $80K          B

In [19]:
# ==========================================
# SEZIONE 2: DEFINIZIONE COLONNE E PREPROCESSING
# ==========================================
print("\n" + "="*80)
print("SEZIONE 2: DEFINIZIONE COLONNE E SETUP PREPROCESSING")
print("="*80)

# Definizione colonne numeriche
variabili_numeriche = [
    'Dependent_count', 'Months_on_book', 'Total_Relationship_Count',
    'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit',
    'Total_Revolving_Bal', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Ct',
    'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio'
]

# Ordini per variabili ordinali
education_order = [['Unknown', 'Uneducated', 'High School', 'College', 
                    'Graduate', 'Post-Graduate', 'Doctorate']]

income_order = [['Unknown', 'Less than $40K', '$40K - $60K', '$60K - $80K', 
                 '$80K - $120K', '$120K +']]

# Variabili categoriche
categoriche_nominali = ['Card_Category', 'Marital_Status']
categoriche_ordinali_edu = ['Education_Level']
categoriche_ordinali_inc = ['Income_Category']

print("\n✓ Variabili numeriche:", len(variabili_numeriche))
print("✓ Variabili categoriche nominali:", len(categoriche_nominali))
print("✓ Variabili categoriche ordinali:", len(categoriche_ordinali_edu) + len(categoriche_ordinali_inc))



SEZIONE 2: DEFINIZIONE COLONNE E SETUP PREPROCESSING

✓ Variabili numeriche: 11
✓ Variabili categoriche nominali: 2
✓ Variabili categoriche ordinali: 2


In [20]:
# Creazione transformers

# se si vuole la standardizzazione rimuovere i commenti 
# dato che usermo il random forest non usiamo la standardizzazione
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

ordinal_edu_transformer = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(
        categories=education_order, 
        handle_unknown='use_encoded_value', 
        unknown_value=-1
    ))
])

ordinal_inc_transformer = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(
        categories=income_order, 
        handle_unknown='use_encoded_value', 
        unknown_value=-1
    ))
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(
        drop='first', 
        handle_unknown='ignore', 
        sparse_output=False
    ))
])

# ColumnTransformer completo
preprocessor = ColumnTransformer(
    transformers=[
        # ('num', numeric_transformer, variabili_numeriche),
        ('ord_edu', ordinal_edu_transformer, categoriche_ordinali_edu),
        ('ord_inc', ordinal_inc_transformer, categoriche_ordinali_inc),
        ('cat', categorical_transformer, categoriche_nominali)
    ],
    remainder='drop'
)

print("\n✓ Preprocessor creato con successo")


✓ Preprocessor creato con successo


In [21]:
# ==========================================
# SEZIONE 3: SPLIT TRAIN-TEST E ENCODING TARGET
# ==========================================
print("\n" + "="*80)
print("SEZIONE 3: SPLIT TRAIN-TEST E ENCODING TARGET")
print("="*80)

# Separazione X e y
X = df.drop(columns=['Attrition_Flag'])
y = df['Attrition_Flag']

# Encoding del target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"\n✓ Target encoding:")
print(f"  {le.classes_[0]} -> 0")
print(f"  {le.classes_[1]} -> 1")

# Split stratificato
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded
)

print(f"\n✓ Split completato:")
print(f"  Train set: {X_train.shape[0]} esempi")
print(f"  Test set: {X_test.shape[0]} esempi")

print(f"\n  Distribuzione train:")
unique_train, counts_train = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique_train, counts_train):
    print(f"    Classe {cls} ({le.classes_[cls]}): {cnt} ({cnt/len(y_train)*100:.2f}%)")

print(f"\n  Distribuzione test:")
unique_test, counts_test = np.unique(y_test, return_counts=True)
for cls, cnt in zip(unique_test, counts_test):
    print(f"    Classe {cls} ({le.classes_[cls]}): {cnt} ({cnt/len(y_test)*100:.2f}%)")


SEZIONE 3: SPLIT TRAIN-TEST E ENCODING TARGET

✓ Target encoding:
  Attrited Customer -> 0
  Existing Customer -> 1

✓ Split completato:
  Train set: 6546 esempi
  Test set: 1637 esempi

  Distribuzione train:
    Classe 0 (Attrited Customer): 1061 (16.21%)
    Classe 1 (Existing Customer): 5485 (83.79%)

  Distribuzione test:
    Classe 0 (Attrited Customer): 265 (16.19%)
    Classe 1 (Existing Customer): 1372 (83.81%)


In [22]:
# ==========================================
# FUNZIONE HELPER: VALUTAZIONE MODELLI
# ==========================================

def evaluate_model(model, X_test_proc, y_test_actual, model_name, le_classes):
    """
    Valuta un modello e restituisce metriche dettagliate
    
    Args:
        model: modello addestrato
        X_test_proc: dati test preprocessati
        y_test_actual: etichette test
        model_name: nome del modello per output
        le_classes: classi del LabelEncoder
    
    Returns:
        dict con tutte le metriche
    """
    print(f"\n{'='*80}")
    print(f"RISULTATI: {model_name}")
    print(f"{'='*80}")
    
    # Predizioni
    y_pred = model.predict(X_test_proc)
    y_pred_proba = model.predict_proba(X_test_proc)[:, 1]
    
    # Confusion Matrix
    cm = confusion_matrix(y_test_actual, y_pred)
    print(f"\nConfusion Matrix:")
    print(f"                  Predicted 0    Predicted 1")
    print(f"Actual 0 (Attr)      {cm[0,0]:6d}         {cm[0,1]:6d}")
    print(f"Actual 1 (Exist)     {cm[1,0]:6d}         {cm[1,1]:6d}")
    
    # Classification Report
    print(f"\n{classification_report(y_test_actual, y_pred, target_names=le_classes)}")
    
    # Metriche chiave
    precision = precision_score(y_test_actual, y_pred)
    recall = recall_score(y_test_actual, y_pred)
    f1 = f1_score(y_test_actual, y_pred)
    balanced_acc = balanced_accuracy_score(y_test_actual, y_pred)
    roc_auc = roc_auc_score(y_test_actual, y_pred_proba)
    
    precision_vals, recall_vals, _ = precision_recall_curve(y_test_actual, y_pred_proba)
    pr_auc = auc(recall_vals, precision_vals)
    
    print(f"{'─'*60}")
    print(f"METRICHE CHIAVE (Focus: Classe Minoritaria)")
    print(f"{'─'*60}")
    print(f"Precision:         {precision:.4f}")
    print(f"Recall:            {recall:.4f}")
    print(f"F1-Score:          {f1:.4f}")
    print(f"Balanced Accuracy: {balanced_acc:.4f}")
    print(f"ROC-AUC:           {roc_auc:.4f}")
    print(f"PR-AUC:            {pr_auc:.4f}")
    
    return {
        'model_name': model_name,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'balanced_acc': balanced_acc,
        'roc_auc': roc_auc,
        'pr_auc': pr_auc
    }

In [23]:
# ==========================================
# ███████████████████████████████████████████████████████████████████████████
# █ APPROCCIO 0: BASELINE - SOLO CLASS WEIGHTING (MODELLO ATTUALE)         █
# ███████████████████████████████████████████████████████████████████████████
print("\n\n" + "="*80)
print("APPROCCIO 0: BASELINE - SOLO CLASS WEIGHTING")
print("="*80)
print("\nDescrizione:")
print("  - Usa tutti i dati di training senza resampling")
print("  - Applica class_weight='balanced' in Random Forest")
print("  - Questo è il modello attuale che vogliamo battere")

# Calcolo class weights
class_weights_baseline = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict_baseline = {i: w for i, w in enumerate(class_weights_baseline)}

print(f"\nClass weights calcolati:")
print(f"  Classe 0 (Attrited):  {class_weight_dict_baseline[0]:.4f}")
print(f"  Classe 1 (Existing):  {class_weight_dict_baseline[1]:.4f}")
print(f"\nDimensione training set: {X_train.shape[0]} esempi")

# Pipeline completa
print("\nCreazione pipeline e training...")
start_time = time.time()

baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        random_state=42,
        n_estimators=400,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=4,
        class_weight=class_weight_dict_baseline,
        n_jobs=-1
    ))
])

baseline_pipeline.fit(X_train, y_train)
baseline_time = time.time() - start_time

print(f"✓ Training completato in {baseline_time:.2f} secondi")

# Preprocessing del test set
X_test_processed = baseline_pipeline.named_steps['preprocessor'].transform(X_test)

# Valutazione
results_baseline = evaluate_model(
    baseline_pipeline.named_steps['classifier'],
    X_test_processed,
    y_test,
    "BASELINE (Class Weighting)",
    le.classes_
)



APPROCCIO 0: BASELINE - SOLO CLASS WEIGHTING

Descrizione:
  - Usa tutti i dati di training senza resampling
  - Applica class_weight='balanced' in Random Forest
  - Questo è il modello attuale che vogliamo battere

Class weights calcolati:
  Classe 0 (Attrited):  3.0848
  Classe 1 (Existing):  0.5967

Dimensione training set: 6546 esempi

Creazione pipeline e training...
✓ Training completato in 0.34 secondi

RISULTATI: BASELINE (Class Weighting)

Confusion Matrix:
                  Predicted 0    Predicted 1
Actual 0 (Attr)         122            143
Actual 1 (Exist)        558            814

                   precision    recall  f1-score   support

Attrited Customer       0.18      0.46      0.26       265
Existing Customer       0.85      0.59      0.70      1372

         accuracy                           0.57      1637
        macro avg       0.51      0.53      0.48      1637
     weighted avg       0.74      0.57      0.63      1637

──────────────────────────────────────

In [24]:
# ==========================================
# ███████████████████████████████████████████████████████████████████████████
# █ APPROCCIO 1: TOMEK LINKS                                               █
# ███████████████████████████████████████████████████████████████████████████
print("\n\n" + "="*80)
print("APPROCCIO 1: TOMEK LINKS")
print("="*80)
print("\nDescrizione:")
print("  - Identifica coppie di esempi (maggioritaria-minoritaria) molto vicini")
print("  - Rimuove l'esempio della classe maggioritaria da queste coppie")
print("  - Obiettivo: pulire il boundary decisionale tra le classi")
print("  - Preserva TUTTI gli esempi della classe minoritaria")

# Preprocessa il training set
X_train_processed = preprocessor.fit_transform(X_train)

print(f"\nDimensione originale: {X_train_processed.shape[0]} esempi")

# Applica Tomek Links
print("Applicazione Tomek Links...")
start_resample = time.time()
tomek = TomekLinks(sampling_strategy='majority')
X_train_tomek, y_train_tomek = tomek.fit_resample(X_train_processed, y_train)
resample_time = time.time() - start_resample

print(f"✓ Resampling completato in {resample_time:.2f} secondi")
print(f"\nDimensione dopo Tomek Links: {X_train_tomek.shape[0]} esempi")
print(f"Esempi rimossi: {X_train_processed.shape[0] - X_train_tomek.shape[0]}")

print(f"\nDistribuzione classi dopo Tomek Links:")
unique, counts = np.unique(y_train_tomek, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Classe {cls} ({le.classes_[cls]}): {cnt} ({cnt/len(y_train_tomek)*100:.2f}%)")

# Ricalcola class weights sul nuovo dataset
class_weights_tomek = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_tomek),
    y=y_train_tomek
)
class_weight_dict_tomek = {i: w for i, w in enumerate(class_weights_tomek)}
print(f"\nNuovi class weights:")
print(f"  Classe 0: {class_weight_dict_tomek[0]:.4f}")
print(f"  Classe 1: {class_weight_dict_tomek[1]:.4f}")

# Training
print("\nTraining modello...")
start_time = time.time()

model_tomek = RandomForestClassifier(
    random_state=42,
    n_estimators=400,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight=class_weight_dict_tomek,
    n_jobs=-1
)

model_tomek.fit(X_train_tomek, y_train_tomek)
tomek_time = time.time() - start_time

print(f"✓ Training completato in {tomek_time:.2f} secondi")

# Valutazione
results_tomek = evaluate_model(
    model_tomek,
    X_test_processed,
    y_test,
    "Tomek Links",
    le.classes_
)




APPROCCIO 1: TOMEK LINKS

Descrizione:
  - Identifica coppie di esempi (maggioritaria-minoritaria) molto vicini
  - Rimuove l'esempio della classe maggioritaria da queste coppie
  - Obiettivo: pulire il boundary decisionale tra le classi
  - Preserva TUTTI gli esempi della classe minoritaria

Dimensione originale: 6546 esempi
Applicazione Tomek Links...
✓ Resampling completato in 0.04 secondi

Dimensione dopo Tomek Links: 6546 esempi
Esempi rimossi: 0

Distribuzione classi dopo Tomek Links:
  Classe 0 (Attrited Customer): 1061 (16.21%)
  Classe 1 (Existing Customer): 5485 (83.79%)

Nuovi class weights:
  Classe 0: 3.0848
  Classe 1: 0.5967

Training modello...
✓ Training completato in 0.31 secondi

RISULTATI: Tomek Links

Confusion Matrix:
                  Predicted 0    Predicted 1
Actual 0 (Attr)         122            143
Actual 1 (Exist)        558            814

                   precision    recall  f1-score   support

Attrited Customer       0.18      0.46      0.26       2

In [25]:
# ==========================================
# ███████████████████████████████████████████████████████████████████████████
# █ APPROCCIO 2: NEARMISS                                                  █
# ███████████████████████████████████████████████████████████████████████████
print("\n\n" + "="*80)
print("APPROCCIO 2: NEARMISS")
print("="*80)
print("\nDescrizione:")
print("  - Seleziona esempi della classe maggioritaria più vicini alla minoritaria")
print("  - Versione 2: seleziona esempi maggioritari vicini ai 3 minoritari più lontani")
print("  - Obiettivo: mantenere i casi 'difficili' vicini al boundary")
print("  - Riduce significativamente il dataset maggioritario")

print(f"\nDimensione originale: {X_train_processed.shape[0]} esempi")

# Applica NearMiss
print("Applicazione NearMiss (version 2)...")
start_resample = time.time()
nearmiss = NearMiss(version=2, n_jobs=-1)
X_train_nm, y_train_nm = nearmiss.fit_resample(X_train_processed, y_train)
resample_time = time.time() - start_resample

print(f"✓ Resampling completato in {resample_time:.2f} secondi")
print(f"\nDimensione dopo NearMiss: {X_train_nm.shape[0]} esempi")
print(f"Esempi rimossi: {X_train_processed.shape[0] - X_train_nm.shape[0]}")

print(f"\nDistribuzione classi dopo NearMiss:")
unique, counts = np.unique(y_train_nm, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Classe {cls} ({le.classes_[cls]}): {cnt} ({cnt/len(y_train_nm)*100:.2f}%)")

# Class weights
class_weights_nm = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_nm),
    y=y_train_nm
)
class_weight_dict_nm = {i: w for i, w in enumerate(class_weights_nm)}
print(f"\nNuovi class weights:")
print(f"  Classe 0: {class_weight_dict_nm[0]:.4f}")
print(f"  Classe 1: {class_weight_dict_nm[1]:.4f}")

# Training
print("\nTraining modello...")
start_time = time.time()

model_nm = RandomForestClassifier(
    random_state=42,
    n_estimators=400,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight=class_weight_dict_nm,
    n_jobs=-1
)

model_nm.fit(X_train_nm, y_train_nm)
nm_time = time.time() - start_time

print(f"✓ Training completato in {nm_time:.2f} secondi")

# Valutazione
results_nm = evaluate_model(
    model_nm,
    X_test_processed,
    y_test,
    "NearMiss",
    le.classes_
)



APPROCCIO 2: NEARMISS

Descrizione:
  - Seleziona esempi della classe maggioritaria più vicini alla minoritaria
  - Versione 2: seleziona esempi maggioritari vicini ai 3 minoritari più lontani
  - Obiettivo: mantenere i casi 'difficili' vicini al boundary
  - Riduce significativamente il dataset maggioritario

Dimensione originale: 6546 esempi
Applicazione NearMiss (version 2)...
✓ Resampling completato in 0.28 secondi

Dimensione dopo NearMiss: 2122 esempi
Esempi rimossi: 4424

Distribuzione classi dopo NearMiss:
  Classe 0 (Attrited Customer): 1061 (50.00%)
  Classe 1 (Existing Customer): 1061 (50.00%)

Nuovi class weights:
  Classe 0: 1.0000
  Classe 1: 1.0000

Training modello...
✓ Training completato in 0.26 secondi

RISULTATI: NearMiss

Confusion Matrix:
                  Predicted 0    Predicted 1
Actual 0 (Attr)         205             60
Actual 1 (Exist)        991            381

                   precision    recall  f1-score   support

Attrited Customer       0.17      0

In [26]:
# ==========================================
# ███████████████████████████████████████████████████████████████████████████
# █ APPROCCIO 3: RANDOM UNDERSAMPLING + ENSEMBLE                          █
# ███████████████████████████████████████████████████████████████████████████
print("\n\n" + "="*80)
print("APPROCCIO 3: RANDOM UNDERSAMPLING + ENSEMBLE")
print("="*80)
print("\nDescrizione:")
print("  - Crea N modelli (N=5) con campionamenti casuali diversi")
print("  - Ogni modello usa TUTTI i minoritari + un subset casuale di maggioritari")
print("  - Predizione finale: soft voting (media delle probabilità)")
print("  - Obiettivo: usare tutti i dati senza perdere informazione")

N_MODELS = 5
n_majority = (y_train == 1).sum()
n_minority = (y_train == 0).sum()
sample_size_majority = n_majority // 2

print(f"\nConfigurazione ensemble:")
print(f"  Numero di modelli: {N_MODELS}")
print(f"  Esempi minoritari per modello: {n_minority} (tutti)")
print(f"  Esempi maggioritari per modello: {sample_size_majority} (~50%)")
print(f"  Dimensione training per modello: {n_minority + sample_size_majority}")

ensemble_models = []
total_ensemble_time = 0

print(f"\nTraining {N_MODELS} modelli...")

for i in range(N_MODELS):
    print(f"\n  Modello {i+1}/{N_MODELS}:")
    
    # Random undersampling con seed diverso
    rus = RandomUnderSampler(
        sampling_strategy={1: sample_size_majority, 0: n_minority},
        random_state=42 + i
    )
    X_train_rus, y_train_rus = rus.fit_resample(X_train_processed, y_train)
    
    print(f"    Resampling: {X_train_rus.shape[0]} esempi")
    
    # Class weights
    class_weights_rus = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train_rus),
        y=y_train_rus
    )
    class_weight_dict_rus = {j: w for j, w in enumerate(class_weights_rus)}
    
    # Training
    start_time = time.time()
    model_rus = RandomForestClassifier(
        random_state=42 + i,
        n_estimators=400,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=4,
        class_weight=class_weight_dict_rus,
        n_jobs=-1
    )
    model_rus.fit(X_train_rus, y_train_rus)
    elapsed = time.time() - start_time
    total_ensemble_time += elapsed
    
    print(f"    Training: {elapsed:.2f}s")
    
    ensemble_models.append(model_rus)

print(f"\n✓ Ensemble completato in {total_ensemble_time:.2f} secondi totali")

# Predizioni ensemble (soft voting)
print("\nCalcolo predizioni ensemble (soft voting)...")
ensemble_proba = np.zeros((len(X_test_processed), 2))

for model in ensemble_models:
    ensemble_proba += model.predict_proba(X_test_processed)

ensemble_proba /= N_MODELS
ensemble_pred = np.argmax(ensemble_proba, axis=1)

# Valutazione manuale (non possiamo usare la funzione helper per ensemble)
print(f"\n{'='*80}")
print(f"RISULTATI: Random Undersampling + Ensemble")
print(f"{'='*80}")

cm = confusion_matrix(y_test, ensemble_pred)
print(f"\nConfusion Matrix:")
print(f"                  Predicted 0    Predicted 1")
print(f"Actual 0 (Attr)      {cm[0,0]:6d}         {cm[0,1]:6d}")
print(f"Actual 1 (Exist)     {cm[1,0]:6d}         {cm[1,1]:6d}")

print(f"\n{classification_report(y_test, ensemble_pred, target_names=le.classes_)}")

precision = precision_score(y_test, ensemble_pred)
recall = recall_score(y_test, ensemble_pred)
f1 = f1_score(y_test, ensemble_pred)
balanced_acc = balanced_accuracy_score(y_test, ensemble_pred)
roc_auc = roc_auc_score(y_test, ensemble_proba[:, 1])

precision_vals, recall_vals, _ = precision_recall_curve(y_test, ensemble_proba[:, 1])
pr_auc = auc(recall_vals, precision_vals)

print(f"{'─'*60}")
print(f"METRICHE CHIAVE (Focus: Classe Minoritaria)")
print(f"{'─'*60}")
print(f"Precision:         {precision:.4f}")
print(f"Recall:            {recall:.4f}")
print(f"F1-Score:          {f1:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"ROC-AUC:           {roc_auc:.4f}")
print(f"PR-AUC:            {pr_auc:.4f}")

results_ensemble = {
    'model_name': 'Random Undersampling + Ensemble',
    'precision': precision,
    'recall': recall,
    'f1': f1,
    'balanced_acc': balanced_acc,
    'roc_auc': roc_auc,
    'pr_auc': pr_auc
}



APPROCCIO 3: RANDOM UNDERSAMPLING + ENSEMBLE

Descrizione:
  - Crea N modelli (N=5) con campionamenti casuali diversi
  - Ogni modello usa TUTTI i minoritari + un subset casuale di maggioritari
  - Predizione finale: soft voting (media delle probabilità)
  - Obiettivo: usare tutti i dati senza perdere informazione

Configurazione ensemble:
  Numero di modelli: 5
  Esempi minoritari per modello: 1061 (tutti)
  Esempi maggioritari per modello: 2742 (~50%)
  Dimensione training per modello: 3803

Training 5 modelli...

  Modello 1/5:
    Resampling: 3803 esempi
    Training: 0.29s

  Modello 2/5:
    Resampling: 3803 esempi
    Training: 0.28s

  Modello 3/5:
    Resampling: 3803 esempi
    Training: 0.27s

  Modello 4/5:
    Resampling: 3803 esempi
    Training: 0.27s

  Modello 5/5:
    Resampling: 3803 esempi
    Training: 0.28s

✓ Ensemble completato in 1.39 secondi totali

Calcolo predizioni ensemble (soft voting)...

RISULTATI: Random Undersampling + Ensemble

Confusion Matrix:
  

In [27]:
# ==========================================
# ███████████████████████████████████████████████████████████████████████████
# █ APPROCCIO 4: SMOTE + UNDERSAMPLING (HYBRID)                           █
# ███████████████████████████████████████████████████████████████████████████
print("\n\n" + "="*80)
print("APPROCCIO 4: SMOTE + UNDERSAMPLING (HYBRID)")
print("="*80)
print("\nDescrizione:")
print("  - SMOTE: genera esempi sintetici della classe minoritaria")
print("  - Undersampling: riduce la classe maggioritaria")
print("  - Obiettivo: bilanciamento moderato (rapporto 2:1)")
print("  - Pro: aumenta variabilità minoritaria, riduce ridondanza maggioritaria")

TARGET_MINORITY = 2500
TARGET_MAJORITY = 5000

print(f"\nTarget configurazione:")
print(f"  Classe minoritaria (con SMOTE): {TARGET_MINORITY}")
print(f"  Classe maggioritaria (undersampled): {TARGET_MAJORITY}")
print(f"  Rapporto finale: {TARGET_MAJORITY/TARGET_MINORITY:.1f}:1")

# Step 1: SMOTE
print(f"\nStep 1: Applicazione SMOTE...")
start_resample = time.time()
smote = SMOTE(sampling_strategy={0: TARGET_MINORITY}, random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train)
smote_time = time.time() - start_resample

print(f"✓ SMOTE completato in {smote_time:.2f} secondi")
print(f"  Classe 0 (minoritaria): {(y_train_smote == 0).sum()}")
print(f"  Classe 1 (maggioritaria): {(y_train_smote == 1).sum()}")

# Step 2: Random Undersampling
print(f"\nStep 2: Applicazione Random Undersampling...")
start_resample = time.time()
rus = RandomUnderSampler(
    sampling_strategy={1: TARGET_MAJORITY, 0: TARGET_MINORITY},
    random_state=42
)
X_train_hybrid, y_train_hybrid = rus.fit_resample(X_train_smote, y_train_smote)
rus_time = time.time() - start_resample

print(f"✓ Undersampling completato in {rus_time:.2f} secondi")
print(f"\nDataset finale:")
print(f"  Classe 0: {(y_train_hybrid == 0).sum()}")
print(f"  Classe 1: {(y_train_hybrid == 1).sum()}")
print(f"  Totale: {len(y_train_hybrid)} esempi")
print(f"  Rapporto: {(y_train_hybrid == 1).sum() / (y_train_hybrid == 0).sum():.2f}:1")

# Class weights
class_weights_hybrid = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_hybrid),
    y=y_train_hybrid
)
class_weight_dict_hybrid = {i: w for i, w in enumerate(class_weights_hybrid)}
print(f"\nClass weights:")
print(f"  Classe 0: {class_weight_dict_hybrid[0]:.4f}")
print(f"  Classe 1: {class_weight_dict_hybrid[1]:.4f}")

# Training
print("\nTraining modello...")
start_time = time.time()

model_hybrid = RandomForestClassifier(
    random_state=42,
    n_estimators=400,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight=class_weight_dict_hybrid,
    n_jobs=-1
)

model_hybrid.fit(X_train_hybrid, y_train_hybrid)
hybrid_time = time.time() - start_time

print(f"✓ Training completato in {hybrid_time:.2f} secondi")

# Valutazione
results_hybrid = evaluate_model(
    model_hybrid,
    X_test_processed,
    y_test,
    "SMOTE + Undersampling",
    le.classes_
)



APPROCCIO 4: SMOTE + UNDERSAMPLING (HYBRID)

Descrizione:
  - SMOTE: genera esempi sintetici della classe minoritaria
  - Undersampling: riduce la classe maggioritaria
  - Obiettivo: bilanciamento moderato (rapporto 2:1)
  - Pro: aumenta variabilità minoritaria, riduce ridondanza maggioritaria

Target configurazione:
  Classe minoritaria (con SMOTE): 2500
  Classe maggioritaria (undersampled): 5000
  Rapporto finale: 2.0:1

Step 1: Applicazione SMOTE...
✓ SMOTE completato in 0.01 secondi
  Classe 0 (minoritaria): 2500
  Classe 1 (maggioritaria): 5485

Step 2: Applicazione Random Undersampling...
✓ Undersampling completato in 0.00 secondi

Dataset finale:
  Classe 0: 2500
  Classe 1: 5000
  Totale: 7500 esempi
  Rapporto: 2.00:1

Class weights:
  Classe 0: 1.5000
  Classe 1: 0.7500

Training modello...
✓ Training completato in 0.33 secondi

RISULTATI: SMOTE + Undersampling

Confusion Matrix:
                  Predicted 0    Predicted 1
Actual 0 (Attr)         118            147
Actual

In [13]:
# ==========================================
# ███████████████████████████████████████████████████████████████████████████
# █ CONFRONTO FINALE E RACCOMANDAZIONE                                    █
# ███████████████████████████████████████████████████████████████████████████
print("\n\n" + "="*80)
print("CONFRONTO FINALE TRA TUTTI GLI APPROCCI")
print("="*80)

# Crea DataFrame con tutti i risultati
results_df = pd.DataFrame([
    results_baseline,
    results_tomek,
    results_nm,
    results_ensemble,
    results_hybrid
])

# Formatta la tabella per output leggibile
print("\n" + "─"*80)
print("TABELLA COMPARATIVA - TUTTE LE METRICHE")
print("─"*80)

# Header
print(f"\n{'Modello':<35} {'Prec':>7} {'Recall':>7} {'F1':>7} {'BalAcc':>7} {'ROC':>7} {'PR-AUC':>7}")
print("─"*80)

# Righe
for _, row in results_df.iterrows():
    print(f"{row['model_name']:<35} "
          f"{row['precision']:>7.4f} "
          f"{row['recall']:>7.4f} "
          f"{row['f1']:>7.4f} "
          f"{row['balanced_acc']:>7.4f} "
          f"{row['roc_auc']:>7.4f} "
          f"{row['pr_auc']:>7.4f}")

print("─"*80)



CONFRONTO FINALE TRA TUTTI GLI APPROCCI

────────────────────────────────────────────────────────────────────────────────
TABELLA COMPARATIVA - TUTTE LE METRICHE
────────────────────────────────────────────────────────────────────────────────

Modello                                Prec  Recall      F1  BalAcc     ROC  PR-AUC
────────────────────────────────────────────────────────────────────────────────
BASELINE (Class Weighting)           0.8506  0.5933  0.6990  0.5268  0.5380  0.8566
Tomek Links                          0.8506  0.5933  0.6990  0.5268  0.5380  0.8566
NearMiss                             0.8639  0.2777  0.4203  0.5256  0.5526  0.8633
Random Undersampling + Ensemble      0.8508  0.5692  0.6821  0.5261  0.5369  0.8596
SMOTE + Undersampling                0.8478  0.5969  0.7006  0.5211  0.5337  0.8574
────────────────────────────────────────────────────────────────────────────────


In [14]:
# ==========================================
# ANALISI: MIGLIORE PER OGNI METRICA
# ==========================================
print("\n" + "="*80)
print("MIGLIORE APPROCCIO PER OGNI METRICA")
print("="*80)

metrics_names = {
    'precision': 'Precision',
    'recall': 'Recall',
    'f1': 'F1-Score',
    'balanced_acc': 'Balanced Accuracy',
    'roc_auc': 'ROC-AUC',
    'pr_auc': 'PR-AUC'
}

for metric_key, metric_name in metrics_names.items():
    best_idx = results_df[metric_key].idxmax()
    best_model = results_df.loc[best_idx, 'model_name']
    best_value = results_df.loc[best_idx, metric_key]
    
    # Mostra anche la baseline per confronto
    baseline_value = results_df.loc[0, metric_key]
    diff = best_value - baseline_value
    diff_pct = (diff / baseline_value * 100) if baseline_value > 0 else 0
    
    print(f"\n{metric_name:20s}: {best_model}")
    print(f"  Valore: {best_value:.4f} (Baseline: {baseline_value:.4f}, Diff: {diff:+.4f} / {diff_pct:+.2f}%)")



MIGLIORE APPROCCIO PER OGNI METRICA

Precision           : NearMiss
  Valore: 0.8639 (Baseline: 0.8506, Diff: +0.0134 / +1.57%)

Recall              : SMOTE + Undersampling
  Valore: 0.5969 (Baseline: 0.5933, Diff: +0.0036 / +0.61%)

F1-Score            : SMOTE + Undersampling
  Valore: 0.7006 (Baseline: 0.6990, Diff: +0.0016 / +0.23%)

Balanced Accuracy   : BASELINE (Class Weighting)
  Valore: 0.5268 (Baseline: 0.5268, Diff: +0.0000 / +0.00%)

ROC-AUC             : NearMiss
  Valore: 0.5526 (Baseline: 0.5380, Diff: +0.0146 / +2.72%)

PR-AUC              : NearMiss
  Valore: 0.8633 (Baseline: 0.8566, Diff: +0.0067 / +0.78%)


In [15]:
# ==========================================
# SCORE AGGREGATO E RACCOMANDAZIONE FINALE
# ==========================================
print("\n" + "="*80)
print("RACCOMANDAZIONE FINALE")
print("="*80)

# Calcola score aggregato pesato
# Peso maggiore su Recall (importante per churn) e F1-Score
print("\nCalcolo score aggregato con pesi:")
print("  - Precision:         25%")
print("  - Recall:            30%  (peso maggiore: critico identificare i churner)")
print("  - F1-Score:          25%")
print("  - ROC-AUC:           20%")

results_df['aggregate_score'] = (
    0.25 * results_df['precision'] +
    0.30 * results_df['recall'] +
    0.25 * results_df['f1'] +
    0.20 * results_df['roc_auc']
)

# Trova il migliore
best_overall_idx = results_df['aggregate_score'].idxmax()
best_overall_model = results_df.loc[best_overall_idx, 'model_name']
best_overall_score = results_df.loc[best_overall_idx, 'aggregate_score']

print(f"\n{'='*80}")
print(f"🏆 MIGLIORE APPROCCIO COMPLESSIVO: {best_overall_model}")
print(f"{'='*80}")
print(f"\nScore aggregato: {best_overall_score:.4f}")

print(f"\nDettaglio metriche:")
best_row = results_df.loc[best_overall_idx]
print(f"  Precision:         {best_row['precision']:.4f}")
print(f"  Recall:            {best_row['recall']:.4f}")
print(f"  F1-Score:          {best_row['f1']:.4f}")
print(f"  Balanced Accuracy: {best_row['balanced_acc']:.4f}")
print(f"  ROC-AUC:           {best_row['roc_auc']:.4f}")
print(f"  PR-AUC:            {best_row['pr_auc']:.4f}")


RACCOMANDAZIONE FINALE

Calcolo score aggregato con pesi:
  - Precision:         25%
  - Recall:            30%  (peso maggiore: critico identificare i churner)
  - F1-Score:          25%
  - ROC-AUC:           20%

🏆 MIGLIORE APPROCCIO COMPLESSIVO: BASELINE (Class Weighting)

Score aggregato: 0.6730

Dettaglio metriche:
  Precision:         0.8506
  Recall:            0.5933
  F1-Score:          0.6990
  Balanced Accuracy: 0.5268
  ROC-AUC:           0.5380
  PR-AUC:            0.8566


In [16]:
# ==========================================
# CONFRONTO CON BASELINE
# ==========================================
print(f"\n{'─'*80}")
print("CONFRONTO CON BASELINE")
print(f"{'─'*80}")

baseline_score = results_df.loc[0, 'aggregate_score']

if best_overall_idx != 0:
    # Non è la baseline
    improvement_score = ((best_overall_score - baseline_score) / baseline_score) * 100
    
    print(f"\nMiglioramento score aggregato: {improvement_score:+.2f}%")
    print(f"\nVariazioni per metrica:")
    
    for metric in ['precision', 'recall', 'f1', 'roc_auc']:
        baseline_val = results_df.loc[0, metric]
        best_val = results_df.loc[best_overall_idx, metric]
        diff = best_val - baseline_val
        diff_pct = (diff / baseline_val * 100) if baseline_val > 0 else 0
        
        symbol = "📈" if diff > 0 else "📉" if diff < 0 else "➡️"
        print(f"  {symbol} {metric.upper():15s}: {baseline_val:.4f} → {best_val:.4f} ({diff_pct:+.2f}%)")
    
    # Decisione finale
    print(f"\n{'='*80}")
    if improvement_score > 1.0:
        print("✅ RACCOMANDAZIONE: ADOTTA IL NUOVO APPROCCIO")
        print(f"{'='*80}")
        print(f"\nIl modello '{best_overall_model}' mostra un miglioramento")
        print(f"significativo ({improvement_score:.2f}%) rispetto alla baseline.")
        print(f"\nVantaggi:")
        if best_row['recall'] > results_df.loc[0, 'recall']:
            print(f"  ✓ Migliore recall: identifica più churner")
        if best_row['precision'] > results_df.loc[0, 'precision']:
            print(f"  ✓ Migliore precision: meno falsi allarmi")
        if best_row['f1'] > results_df.loc[0, 'f1']:
            print(f"  ✓ Migliore F1: miglior bilanciamento complessivo")
    elif improvement_score > 0:
        print("⚠️  RACCOMANDAZIONE: MIGLIORAMENTO MARGINALE")
        print(f"{'='*80}")
        print(f"\nIl modello '{best_overall_model}' mostra un miglioramento")
        print(f"modesto ({improvement_score:.2f}%) rispetto alla baseline.")
        print(f"\nConsigli:")
        print(f"  - Valuta se il miglioramento giustifica la complessità aggiuntiva")
        print(f"  - Considera il tempo di training e la manutenibilità")
        print(f"  - Effettua validazione incrociata per confermare i risultati")
    else:
        print("❌ RACCOMANDAZIONE: MANTIENI LA BASELINE")
        print(f"{'='*80}")
        print(f"\nNessun miglioramento rilevato. La baseline rimane la scelta migliore.")
else:
    # La baseline è la migliore
    print("✅ RACCOMANDAZIONE: MANTIENI LA BASELINE")
    print(f"{'='*80}")
    print(f"\nLa BASELINE (solo class weighting) rimane il miglior approccio!")
    print(f"\nMotivi:")
    print(f"  ✓ Performance ottimali senza complessità aggiuntiva")
    print(f"  ✓ Usa tutti i dati disponibili")
    print(f"  ✓ Più semplice da mantenere e spiegare")
    print(f"  ✓ Nessun rischio di perdita di informazione")


────────────────────────────────────────────────────────────────────────────────
CONFRONTO CON BASELINE
────────────────────────────────────────────────────────────────────────────────
✅ RACCOMANDAZIONE: MANTIENI LA BASELINE

La BASELINE (solo class weighting) rimane il miglior approccio!

Motivi:
  ✓ Performance ottimali senza complessità aggiuntiva
  ✓ Usa tutti i dati disponibili
  ✓ Più semplice da mantenere e spiegare
  ✓ Nessun rischio di perdita di informazione


In [17]:
# ==========================================
# ANALISI AGGIUNTIVA: TUTTI GLI SCORE AGGREGATI
# ==========================================
print(f"\n{'─'*80}")
print("CLASSIFICA COMPLETA (per score aggregato)")
print(f"{'─'*80}")

results_sorted = results_df.sort_values('aggregate_score', ascending=False)
for rank, (idx, row) in enumerate(results_sorted.iterrows(), 1):
    medal = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉" if rank == 3 else f"{rank}."
    print(f"\n{medal} {row['model_name']}")
    print(f"   Score: {row['aggregate_score']:.4f} | "
          f"F1: {row['f1']:.4f} | "
          f"Recall: {row['recall']:.4f} | "
          f"Precision: {row['precision']:.4f}")


────────────────────────────────────────────────────────────────────────────────
CLASSIFICA COMPLETA (per score aggregato)
────────────────────────────────────────────────────────────────────────────────

🥇 BASELINE (Class Weighting)
   Score: 0.6730 | F1: 0.6990 | Recall: 0.5933 | Precision: 0.8506

🥈 Tomek Links
   Score: 0.6730 | F1: 0.6990 | Recall: 0.5933 | Precision: 0.8506

🥉 SMOTE + Undersampling
   Score: 0.6729 | F1: 0.7006 | Recall: 0.5969 | Precision: 0.8478

4. Random Undersampling + Ensemble
   Score: 0.6614 | F1: 0.6821 | Recall: 0.5692 | Precision: 0.8508

5. NearMiss
   Score: 0.5149 | F1: 0.4203 | Recall: 0.2777 | Precision: 0.8639


In [ ]:
# ==========================================
# SALVATAGGIO RISULTATI (OPZIONALE)
# ==========================================
print(f"\n{'='*80}")
print("ESPORTAZIONE RISULTATI")
print(f"{'='*80}")

# Salva risultati in CSV
output_filename = 'confronto_approcci_risultati.csv'
results_df.to_csv(output_filename, index=False)
print(f"\n✓ Risultati salvati in: {output_filename}")

print(f"\n{'='*80}")
print("ANALISI COMPLETATA!")
print(f"{'='*80}")
print("\nProssimi passi suggeriti:")
print("  1. Se hai adottato un nuovo approccio, valida con cross-validation")
print("  2. Testa su un ulteriore test set o dati più recenti")
print("  3. Analizza gli esempi misclassificati per capire pattern")
print("  4. Considera feature engineering se nessun approccio ha migliorato significativamente")
print(f"\n{'='*80}")